# ChibiCreate — MODEL EVALUATION: FLUX.2 [klein] 4B

Avalia o candidato **FLUX.2 klein 4B** no experimento padronizado:
`waifu_001 → chibi full body`, seed 42, duas execuções.

---

## Por que este candidato primeiro

| | Qwen-Image-Edit-2511 | FLUX.2 klein 4B |
|---|---|---|
| Params | 20.4B | **3.9B** |
| Pesos (diffusion) | 20.5 GB | **7.75 GB** |
| Download total | ~30 GB | **~16 GB** |
| VRAM declarada | ~24 GB | **~13 GB** |
| Colab **gratuito** (T4 15 GB) | não cabe | **provavelmente cabe** |
| Licença | Apache-2.0 | Apache-2.0 |
| Multi-reference | sim | sim (nativo) |
| Passos | 20 | **4** (destilado) |

O klein é menor, mais barato e cabe onde o Qwen não cabe. Isso **não** o
torna melhor: só torna barato descobrir se ele já basta.

> **⚠️ 4B é Apache-2.0. A variante 9B é NÃO-COMERCIAL.** Confirmado pela BFL
> no model card. Usamos exclusivamente a 4B.

### Diferenças que importam

O klein é **destilado**: roda com `cfg = 1.0` e **4 passos**. Aplicar o
`cfg 2.5` do Qwen produziria imagem lavada — e uma avaliação injusta do
candidato. Isso está em `config/environments/colab_flux2.yaml`, não
hardcoded.

O prompt negativo não existe de forma nativa no FLUX (usa-se
`ConditioningZeroOut`). O recipe registra o campo, mas ele **não alimenta o
grafo** — o recipe descreve o que de fato rodou.


---

## Célula 1 — detectar GPU (portão)

Threshold menor que o do Qwen porque o modelo é menor. Continua sendo um
portão real: se não couber, para.


In [ ]:
import sys, json

WEIGHTS_GB = 7.75      # flux-2-klein-4b.safetensors
MIN_VRAM_GB = 13.0     # model card oficial: '~13GB VRAM'

try:
    import torch
except ImportError:
    torch = None

print('=' * 62)
print('AMBIENTE COLAB — detectado, nao presumido')
print('=' * 62)
print('Python :', sys.version.split()[0])
print('Torch  :', torch.__version__ if torch else 'ausente')

if torch is None or not torch.cuda.is_available():
    print('CUDA   : INDISPONIVEL')
    print('COLAB_GPU_INSUFFICIENT — nenhuma GPU CUDA.')
    print('Runtime -> Alterar tipo de ambiente de execucao -> GPU')
    raise SystemExit('FASE 3B permanece BLOCKED')

props = torch.cuda.get_device_properties(0)
total_gb = props.total_memory / 1024 ** 3
free_gb = torch.cuda.mem_get_info()[0] / 1024 ** 3

GPU_INFO = {
    'name': props.name,
    'vram_total_gb': round(total_gb, 2),
    'vram_free_gb': round(free_gb, 2),
    'cuda': torch.version.cuda,
    'capability': '{}.{}'.format(props.major, props.minor),
    'torch': torch.__version__,
    'python': sys.version.split()[0],
    'bf16_supported': props.major >= 8,
}
print('GPU    :', GPU_INFO['name'])
print('VRAM   : {:.2f} GB total / {:.2f} GB livre'.format(total_gb, free_gb))
print('CUDA   :', GPU_INFO['cuda'], '| capability', GPU_INFO['capability'])
print('Necessario : ~{:.0f} GB'.format(MIN_VRAM_GB))
print()

if not GPU_INFO['bf16_supported']:
    print('AVISO: sem bf16 nativo (capability < 8.0, ex. T4).')
    print('O ComfyUI cai para fp16. Pode funcionar; registre o resultado.')
    print()

if total_gb < MIN_VRAM_GB:
    print('=' * 62)
    print('COLAB_GPU_INSUFFICIENT')
    print('=' * 62)
    print('{} tem {:.1f} GB; o FLUX.2 klein 4B precisa de ~{:.0f} GB.'
          .format(GPU_INFO['name'], total_gb, MIN_VRAM_GB))
    print('PARE. Nao usar CPU, nao trocar de modelo, nao quantizar por conta.')
    raise SystemExit('COLAB_GPU_INSUFFICIENT')

print('GPU ADEQUADA — pode prosseguir.')
json.dump(GPU_INFO, open('/content/gpu_info.json', 'w'), indent=2)


---

## Célula 2 — repositório + ComfyUI


In [ ]:
%cd /content
!git clone --branch arena/01a07ece-chibicreate https://github.com/BloomRX/ChibiCreate.git 2>/dev/null || echo 'ja clonado'
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>/dev/null || echo 'ja clonado'
!pip install -q pyyaml pillow numpy huggingface_hub
!pip install -q -r /content/ComfyUI/requirements.txt

import subprocess, hashlib, pathlib
COMFY_COMMIT = subprocess.check_output(['git','rev-parse','HEAD'],
                                       cwd='/content/ComfyUI').decode().strip()
print('ComfyUI commit:', COMFY_COMMIT)

ref = pathlib.Path('/content/ChibiCreate/characters/waifu_001/reference/full_body.png')
h = hashlib.sha256(ref.read_bytes()).hexdigest()
EXPECTED = '2fdcd5f428f5980d63e31d4bf4a67aecbc11c1b101c19ca75f819db616cb8177'
print('input sha256:', h)
print('confere     :', 'SIM' if h == EXPECTED else 'NAO — investigar')


---

## Célula 3 — baixar os modelos (~16 GB)

Três arquivos de `Comfy-Org/vae-text-encorder-for-flux-klein-4b`, Apache-2.0.

> **Nome importa:** `flux-2-klein-4b` (destilado), **não**
> `flux-2-klein-base-4b`. Mesmo tamanho, modelos diferentes.

Se a VRAM estiver apertada, troque o text encoder pela variante `fp4`
(3.85 GB em vez de 8.04 GB) — comentada no código.


In [ ]:
from huggingface_hub import hf_hub_download
import pathlib, shutil

M = pathlib.Path('/content/ComfyUI/models')
REPO = 'Comfy-Org/vae-text-encorder-for-flux-klein-4b'
REV  = '5f526678002e43af5551dadb73ce2e8c91b43afe'

DOWNLOADS = [
    ('split_files/diffusion_models/flux-2-klein-4b.safetensors',
     M / 'diffusion_models',
     'ec3d4e733a771f61c052fb4856c48b336c55eaf2c65487c2a1faeb9bbda7a343'),
    ('split_files/text_encoders/qwen_3_4b.safetensors',
     M / 'text_encoders',
     '6c671498573ac2f7a5501502ccce8d2b08ea6ca2f661c458e708f36b36edfc5a'),
    # Alternativa fp4 para GPU apertada — troque a linha acima por esta:
    # ('split_files/text_encoders/qwen_3_4b_fp4_flux2.safetensors',
    #  M / 'text_encoders',
    #  '3eab03a77adb0ee5304a4e677d5c10ac22f9049c1d7c894adca4f8bb39206ca8'),
    ('split_files/vae/flux2-vae.safetensors',
     M / 'vae',
     '868fe7b343cc8f3a19dbcfcafbc3d5f888802be3f89bd81b65b3621a066ce8f3'),
]

MODEL_RECORD = []
for remote, dest, expected in DOWNLOADS:
    dest.mkdir(parents=True, exist_ok=True)
    fname = remote.split('/')[-1]
    target = dest / fname
    if target.exists():
        print('ja existe:', fname)
    else:
        print('baixando :', fname)
        got = hf_hub_download(repo_id=REPO, revision=REV, filename=remote)
        shutil.copy(got, target)
    MODEL_RECORD.append({'file': fname, 'repo': REPO, 'revision': REV,
                         'license': 'Apache-2.0',
                         'size_bytes': target.stat().st_size,
                         'sha256_expected': expected})
!df -h /content | tail -1


## Célula 3b — conferir SHA256


In [ ]:
import hashlib, pathlib, json

def sha256_of(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

ok = True
for rec in MODEL_RECORD:
    hit = list(pathlib.Path('/content/ComfyUI/models').rglob(rec['file']))[0]
    actual = sha256_of(hit)
    rec['sha256_actual'] = actual
    rec['sha256_match'] = actual == rec['sha256_expected']
    ok &= rec['sha256_match']
    print(('OK   ' if rec['sha256_match'] else 'FALHA'), rec['file'])

json.dump(MODEL_RECORD, open('/content/model_record.json', 'w'), indent=2)
if not ok:
    raise SystemExit('SHA256 divergente — PARE e reporte.')
print('Pesos conferem com models.lock.yaml.')


---

## Célula 4 — iniciar o ComfyUI


In [ ]:
import subprocess, time, urllib.request, json

LOG = open('/content/comfyui.log', 'w')
proc = subprocess.Popen(['python', 'main.py', '--listen', '127.0.0.1',
                         '--port', '8188'],
                        cwd='/content/ComfyUI', stdout=LOG,
                        stderr=subprocess.STDOUT)

print('subindo ComfyUI...')
for i in range(120):
    time.sleep(5)
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/system_stats',
                                    timeout=5) as r:
            stats = json.load(r)
        print('no ar apos ~{}s'.format((i + 1) * 5))
        break
    except Exception:
        if proc.poll() is not None:
            print(open('/content/comfyui.log').read()[-3000:])
            raise SystemExit('ComfyUI morreu ao iniciar')
else:
    print(open('/content/comfyui.log').read()[-3000:])
    raise SystemExit('ComfyUI nao respondeu em 10 min')

print(json.dumps(stats.get('system', {}), indent=2))


---

## Célula 5 — FASE A: preflight

**Portão.** Este é o momento em que descobrimos se o workflow do FLUX que
escrevemos bate com os nodes reais do ComfyUI — ele nunca foi executado
(`[TEST REQUIRED]`).

Se der `WORKFLOW_INCOMPATIBLE`, **copie o bloco `node/expected/actual`**:
é o dado exato para corrigir o grafo com evidência.


In [ ]:
import os
os.environ['CHIBI_COMFY_URL'] = 'http://127.0.0.1:8188'
%cd /content/ChibiCreate

!python -m scripts.chibi.cli comfy status --env colab_flux2
print('=' * 62)
!python -m scripts.chibi.cli comfy preflight --env colab_flux2
print('=' * 62)
!python -m scripts.chibi.cli comfy validate --env colab_flux2 --workflow experimental/flux2_klein_edit


---

## Célula 6 — execução 1 de 2


In [ ]:
PROMPT = ('Transform this character into a clean stylized chibi full-body '
          'character, preserving the same identity, black hair, red eyes, '
          'horns, black outfit, long black cape and golden ornaments.')

%cd /content/ChibiCreate
!python -m scripts.chibi.cli experiment model-eval \
    --model flux2-klein --character waifu_001 --seed 42 --prompt "$PROMPT"


## Célula 7 — execução 2 de 2 (parâmetros idênticos)


In [ ]:
%cd /content/ChibiCreate
!python -m scripts.chibi.cli experiment model-eval \
    --model flux2-klein --character waifu_001 --seed 42 --prompt "$PROMPT"


---

## Célula 8 — resultados

Hash diferente entre as duas é resultado válido, não falha.


In [ ]:
import pathlib, json
from PIL import Image

base = pathlib.Path('experiments/model_eval/flux2_klein_4b')
runs = sorted(base.glob('run_*'))
print('execucoes:', [r.name for r in runs])

for r in runs:
    recipe = json.load(open(r / 'recipe.json'))
    print()
    print('===', r.name, '=' * 40)
    for k in ('artifact_sha256', 'output_sha256', 'seed', 'execution_time',
              'cuda', 'comfyui_version'):
        if k in recipe:
            print('  {:18} {}'.format(k, recipe[k]))
    print('  parameters        ', recipe.get('parameters'))
    print('  gpu               ', (recipe.get('gpu') or {}).get('name'))
    img = r / 'output.png'
    if img.exists():
        display(Image.open(img))

if len(runs) >= 2:
    a, b = runs[0], runs[1]
    get_ipython().system(
        'python -m scripts.chibi.cli experiment compare {} {}'.format(a, b))


---

## Célula 9 — metadata da sessão


In [ ]:
import json, pathlib

meta = {
    'infrastructure': 'google_colab',
    'infrastructure_status': 'EXPERIMENTAL_TEMPORARY',
    'candidate': 'flux2_klein_4b',
    'gpu': json.load(open('/content/gpu_info.json')),
    'models': json.load(open('/content/model_record.json')),
    'comfyui_commit': COMFY_COMMIT,
    'custom_nodes': [],
    'custom_nodes_note': 'Nenhum. Multi-reference e nativo (ReferenceLatent).',
    'cost': 0,
    'cost_note': 'No direct GPU cost observed in this session.',
    'cost_warning': 'Nao extrapolar para custo de producao.',
    'limitations': [
        'Sessao efemera: /content e perdido ao desconectar.',
        'GPU nao garantida entre sessoes.',
        'Colab NAO e infraestrutura permanente do projeto.',
    ],
}
p = pathlib.Path('/content/ChibiCreate/experiments/model_eval/colab_session.json')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))


---

## Célula 10 — salvar (a sessão é efêmera)


---

## RUN 003 — multi-referência (3 refs)

A **única** variável em relação aos runs 001/002 é o número de referências:
`full_body` → `full_body + face + outfit`.

Modelo, prompt, seed, cfg, steps, sampler e resolução são idênticos. Isso é
garantido pelo workflow `v2`, que difere do `v1` apenas por encadear mais dois
`ReferenceLatent` (há teste automatizado travando esses parâmetros).

Antes de executar, a célula valida o `v2` contra `/object_info`. **Se a
validação falhar, pare** — não edite o workflow para "fazer passar".


In [ ]:
%cd /content/ChibiCreate
# Validacao OBRIGATORIA do v2 contra o servidor real.
!python -m scripts.chibi.cli comfy validate --env colab_flux2 \
    --workflow experimental/flux2_klein_edit

# Confirma a cadeia das 3 referencias ANTES de gastar GPU.
import json
wf = json.load(open('workflows/experimental/flux2_klein_edit/v2.json'))
n = {k: v for k, v in wf.items() if not k.startswith('_')}

def upstream(nid, seen=None):
    seen = seen if seen is not None else set()
    if nid in seen:
        return seen
    seen.add(nid)
    for v in n[nid]['inputs'].values():
        if isinstance(v, list) and len(v) == 2 and isinstance(v[0], str):
            upstream(v[0], seen)
    return seen

loads = {k for k, v in n.items() if v['class_type'] == 'LoadImage'}
reach = upstream(n['10']['inputs']['positive'][0])
print('LoadImage:', len(loads), '| alcancam KSampler.positive:', len(loads & reach))
assert loads <= reach, 'PARE: alguma referencia nao condiciona nada'
print('OK — as 3 referencias entram no condicionamento')

In [ ]:
%cd /content/ChibiCreate
# RUN 003 — mesmas condicoes, 3 referencias.
!python -m scripts.chibi.cli experiment model-eval \
    --model flux2-klein --character waifu_001 --seed 42 \
    --workflow-version v2 \
    --ref reference/face.png --ref reference/outfit.png \
    --prompt "$PROMPT"

# Confere o que o recipe registrou.
import json, pathlib
runs = sorted(pathlib.Path('experiments/model_eval/flux2_klein_4b').glob('run_*'))
r = json.load(open(runs[-1] / 'recipe.json'))
print('\n', runs[-1].name, '| reference_count:', r.get('reference_count'))
for ref in r.get('references', []):
    print('  ', ref['role'], ref['file'], ref['sha256'][:16])

In [ ]:
import shutil
shutil.make_archive('/content/model_eval_flux2', 'zip',
                    '/content/ChibiCreate/experiments')
from google.colab import files
files.download('/content/model_eval_flux2.zip')


---

## PARE AQUI

Duas execuções reais do FLUX.2 klein 4B — fim do escopo.

**Não testar o Qwen agora.** Essa decisão vem depois da revisão humana.

### [HUMAN REVIEW REQUIRED]

Preencha `docs/model-eval/ficha-avaliacao.md`:

HAIR · FACE · EYES · HORNS · OUTFIT · CAPE · ACCESSORIES · SILHOUETTE ·
OVERALL

E conclua com **uma** marca: `PROMISING` · `INSUFFICIENT` · `BLOCKED`.

Depois disso decidimos: (A) testar o Qwen também, ou (B) o klein já basta.

Nada de LoRA, ControlNet, 8 candidatos, `master.png`, Flow 02 ou animação.
